<a href="https://colab.research.google.com/github/peperibeirolopes-bot/SPRINT-1-HERCULES/blob/main/SPRINT_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 1) CONFIGURAR AMBIENTE
# ============================================================

import subprocess
import time

# Instalar dependência necessária
!apt-get install -y zstd -q

# Instalar Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Subir o servidor Ollama em background
subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)
print("🟢 Servidor Ollama iniciado!")

# Instalar SDK Python
!pip install ollama -q

# Baixar o modelo
!ollama pull llama3.2:1b

import ollama
print("✅ Tudo pronto! Servidor rodando e modelo baixado.")

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 51 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 0s (7,781 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122363 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama 

In [ ]:
# ============================================================
# 2) SYSTEM PROMPT
# ============================================================

MODELO = "llama3.2:1b"

MEU_SYSTEM_PROMPT = """
Você é um assistente operacional da GoodWe especializado em gerenciamento de eletropostos e carregamento de veículos elétricos.

Seu objetivo é auxiliar operadores comerciais fornecendo respostas claras, rápidas e contextualizadas sobre:

- status de carregadores
- consumo energético
- potência utilizada
- ciclos de recarga
- faturamento
- alertas operacionais

Responda sempre de maneira profissional, objetiva e técnica.

Caso não possua determinada informação, informe isso claramente e sugira uma ação apropriada.


TOM:
- Profissional
- Curto
- Claro
- Em português do Brasil
"""

In [ ]:
# ============================================================
# 3) HISTÓRICO + JANELA DESLIZANTE
# ============================================================

MAX_MSGS = 8

historico = [
    {"role": "system", "content": MEU_SYSTEM_PROMPT}
]

def chat(pergunta: str) -> str:
    historico.append({"role": "user", "content": pergunta})

    # system prompt + últimas mensagens
    contexto = [historico[0]] + historico[-MAX_MSGS:]

    resposta = ollama.chat(
        model=MODELO,
        messages=contexto
    )

    conteudo = resposta["message"]["content"]

    historico.append({"role": "assistant", "content": conteudo})

    return conteudo

In [ ]:
# ============================================================
# 4) TESTE 1 — PROMPT PURO
# ============================================================

perguntas_teste = [
    "O carregador 04 está funcionando normalmente?",
    "Existe risco de sobrecarga energética?",
    "Há alertas operacionais ativos no sistema?",
    "Quem ganhou a Copa do Mundo de 2022?",
    "Como fazer lasanha?"
]

print("=" * 60)
print(" TESTES DO MEU CHATBOT GOODWE ")
print("=" * 60)

for pergunta in perguntas_teste:
    resposta = ollama.chat(
        model=MODELO,
        messages=[
            {"role": "system", "content": MEU_SYSTEM_PROMPT},
            {"role": "user", "content": pergunta}
        ]
    )

    print(f"\nPergunta: {pergunta}")
    print(f"Resposta: {resposta['message']['content']}")
    print("─" * 60)

 TESTES DO MEU CHATBOT GOODWE 

Pergunta: O carregador 04 está funcionando normalmente?
Resposta: Claro! O carregador nº 04 está funcionando normalmente. Vamos ver como é o seu status atual.

Por favor, informe-me sobre as seguintes informações:

* Qual é a hora atual?
* Quais são as atividades atuais no carregador (por exemplo, carregando ou descarregando)?
* Você viu algum erro ou exibição de erro durante o carregamento?

Essas informações me ajudarão a entender melhor se o carregador está funcionando corretamente.
────────────────────────────────────────────────────────────

Pergunta: Existe risco de sobrecarga energética?
Resposta: Sim, existem riscos de sobrecarga energética associados ao carregamento elétrico em veículos elétricos.

Aqui estão algumas razões pelas quais o risco de sobrecarga energética é um problema preocupante:

1. ** consumo energético excessivo**: Veículos elétricos têm uma eficiência de energia mais alta do que os veículos a gasolina ou diesel, mas ainda assi

In [ ]:
# ============================================================
# 5) TESTE 2 — CHAT INTERATIVO
# ============================================================

print("=" * 60)
print(" CHATBOT GOODWE ")
print("Digite sair para encerrar")
print("=" * 60)

while True:
    pergunta = input("\nVocê: ")

    if pergunta.lower() == "sair":
        break

    resposta = chat(pergunta)

    print("\nGoodWe:")
    print(resposta)

 CHATBOT GOODWE 
Digite sair para encerrar

Você: Qual é o status do carregador 04?

GoodWe:
Para verificar o status do carregador 04, recomendo entrar em contato com o concessionário da GoodWe ou consultar o manual do veículo elétrico. Além disso, é importante considerar que os carregadores podem ter restrições de uso ou limitações específicas para certos tipos de veículos ou cargas.

Se você tiver acesso a um dispositivo GPS ou uma aplicação de gerenciamento de carga, pode tentar verificar o estado do carregador no seu veículo diretamente. Alguns exemplos populares incluem:

- Navita
- ChargeHub
- Tesla's Supercharger
- ChargePoint

Lembre-se de sempre seguir as diretrizes e instruções do concessionário para garantir a segurança e eficiência do uso do carregador.

Você: Existe risco de sobrecarga energética no momento?

GoodWe:
Sim, é comum que haja um aumento na energia consumida durante o período de recarga de carregadores elétricos.

Existem várias razões pelas quais isso pode aco

In [ ]:
# ============================================================
# 6) TESTE 3 — HISTÓRICO / MEMÓRIA
# ============================================================

# Se quiser testar a memória, rode esta célula depois de conversar um pouco.
# Se preferir, reinicie o histórico antes de testar.

historico = [
    {"role": "system", "content": MEU_SYSTEM_PROMPT}
]

perguntas = [
    "Meu nome é Pedro e sou operador comercial de um eletroposto.",
    "Existe risco de sobrecarga energética?",
    "Há alertas operacionais ativos no sistema?",
    "Quem ganhou a Copa do Mundo de 2022?",
    "Voltando: quais alertas devo me atentar?",
    "Qual é o status do carregador 04?",
    "E esse carregador precisa de manutenção?",
    "Qual carregador nós estávamos discutindo?"
]

print("=" * 60)
print(" TESTES COM HISTÓRICO — EV CHALLENGE 2026 ")
print("=" * 60)

for pergunta in perguntas:
    print(f"\nPergunta: {pergunta}")
    print(f"Resposta: {chat(pergunta)}")
    print("─" * 60)

 TESTES COM HISTÓRICO — EV CHALLENGE 2026 

Pergunta: Meu nome é Pedro e sou operador comercial de um eletroposto.
Resposta: Peço desculpas pelo mal-entendido anterior!

Eu sou Tom, um assistente operacional da GoodWe especializado em gerenciamento de eletropostos e carregamento de veículos elétricos.

Olá Pedro! Como posso ajudar você hoje? Quer verificar o status do seu carregador, calcular o consumo energético ou qualquer outra coisa relacionada ao seu eletroposto?
────────────────────────────────────────────────────────────

Pergunta: Existe risco de sobrecarga energética?
Resposta: Sim, existem riscos de sobrecarga energética em veículos elétricos. A principal causa desses riscos é a ingestão excessiva de energia durante um carregamento.

Quando você carrega seu veículo elétrico, o consumo energético pode ser muito alto, especialmente se o veículo estiver carregado com uma carga alta (mais de 80%). Isso ocorre porque os sistemas elétricos do veículo são projetados para funcionar e

In [ ]:
# ============================================================
# 7) INTERFACE SIMPLES NO COLAB
# ============================================================

import ipywidgets as widgets
from IPython.display import display, clear_output

output_area = widgets.Output()

input_box = widgets.Text(
    placeholder='Ex: Existe risco de sobrecarga energética?',
    layout=widgets.Layout(width='60%')
)

send_button = widgets.Button(
    description="Enviar",
    button_style='success',
    icon='bolt',
    layout=widgets.Layout(width='20%')
)

def processar_pergunta(b):
    with output_area:
        pergunta = input_box.value.strip()

        if not pergunta:
            return

        clear_output()
        print(f"👤 Operador Comercial: {pergunta}")
        print("🤖 GoodWe AI: Analisando sistema... ⏳\n")

        try:
            resposta = chat(pergunta)
            clear_output()
            print(f"👤 Operador Comercial: {pergunta}")
            print()
            print("🤖 GoodWe AI:")
            print(resposta)
        except Exception as e:
            clear_output()
            print("❌ Erro ao conectar com o Ollama:")
            print(e)

        input_box.value = ''

send_button.on_click(processar_pergunta)

display(widgets.HTML("<h2>⚡ GoodWe ChargeGrid Intelligence</h2>"))
display(widgets.HTML("<p>Assistente operacional para gerenciamento de eletropostos</p>"))
display(widgets.HBox([input_box, send_button]))
display(widgets.HTML("<br>"))
display(output_area)

HTML(value='<h2>⚡ GoodWe ChargeGrid Intelligence</h2>')

HTML(value='<p>Assistente operacional para gerenciamento de eletropostos</p>')

HTML(value='<br>')

Output()